In [1]:
import numpy as np
from sklearn.externals.array_api_extra.testing import override

from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

Current RAM usage: 298.89 MB


In [2]:
from sklearn.decomposition import IncrementalPCA

n_components = 15
def data_pipeline(plane):
    _x_train, _y_train = get_data(train_full, "train_series", plane)
    _x_test, _y_test = get_data(test_full, "train_series", plane)

    _x_train = np.reshape(_x_train, shape=(_x_train.shape[0], _x_train.shape[1] * _x_train.shape[2] *  _x_train.shape[3]))
    _x_test = np.reshape(_x_test, shape=(_x_test.shape[0], _x_test.shape[1] * _x_test.shape[2] *  _x_test.shape[3]))

    n_batches = floor(_x_train.shape[0] / n_components)
    inc_pca = IncrementalPCA(n_components=n_components)

    for X_batch in np.array_split(_x_train, n_batches):
        inc_pca.partial_fit(X_batch)

    _x_train = inc_pca.transform(_x_train)
    _x_test = inc_pca.transform(_x_test)
    
    return _x_train, _y_train, _x_test, _y_test

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

for p in planes:
    x_train, y_train, x_test, y_test = data_pipeline(p)

    model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    model.fit(x_train, y_train)
    pred = model.predict(x_test)

    scores = roc_auc_score(y_test, pred, average=None)
    print(f"{p} Model Score: {np.mean(scores)}")
    for i in range(len(target_columns)):
        print(f"\t{target_columns[i]}: {scores[i]}")
    print("--------")


Sagittal Model Score: 0.5760435166685166
	ACL: 0.6142857142857143
	MCL: 0.5
	Medial Meniscus: 0.8333333333333333
	Lateral Meniscus: 0.6071428571428572
	Medial OA: 0.4326923076923077
	Lateral OA: 0.6166666666666667
	PF OA: 0.7211538461538461
	Effusion: 0.4652777777777778
	Synovitis: 0.6136363636363635
	Baker's: 0.5499999999999999
	Contusion: 0.4583333333333333
	Fracture: 0.5
--------
Axial Model Score: 0.5451388888888888
	ACL: 0.5
	MCL: 0.5
	Medial Meniscus: 0.6666666666666666
	Lateral Meniscus: 0.35
	Medial OA: 0.575
	Lateral OA: 0.4166666666666667
	PF OA: 0.875
	Effusion: 0.325
	Synovitis: 0.8333333333333333
	Baker's: 0.5
	Contusion: 0.5
	Fracture: 0.5
--------
Coronal Model Score: 0.4916666666666667
	ACL: 0.3857142857142857
	MCL: 0.5
	Medial Meniscus: 0.6857142857142857
	Lateral Meniscus: 0.17142857142857146
	Medial OA: 0.5857142857142856
	Lateral OA: 0.3
	PF OA: 0.7285714285714286
	Effusion: 0.8
	Synovitis: 0.44285714285714284
	Baker's: 0.3
	Contusion: 0.5
	Fracture: 0.5
--------


In [16]:
class LogRegModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, n_comp=15):
        self.n_comp = n_comp
        self.inc_pca = IncrementalPCA(n_components=n_comp)
        self.model = OneVsRestClassifier(LogisticRegression(max_iter=1000))
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression)

    @override
    def train(self, x: np.ndarray, y: np.ndarray):
        print(x.shape)
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        n_batches = floor(x.shape[0] / self.n_comp)

        for X_batch in np.array_split(x, n_batches):
            self.inc_pca.partial_fit(X_batch)

        x_reduced = self.inc_pca.transform(x)
        self.model.fit(x_reduced, y)

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], x.shape[1] * x.shape[2] * x.shape[3]))
        x_reduced = self.inc_pca.transform(x)
        return self.model.predict(x_reduced)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:

        x = np.reshape(x, shape=(1, x.shape[0] * x.shape[1] * x.shape[2]))
        x_reduced = self.inc_pca.transform(x)
        pred_ = self.model.predict(x_reduced)
        return np.reshape(pred_, shape=(pred_.shape[1]))


In [5]:
# Depth 1 ensemble
ensemble = [LogRegModel(p) for p in planes]
scores = Model.get_ensemble_auc_score(ensemble, 1)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

                                               index         0    1         2  \
0  1.2.826.0.1.3680043.8.498.10306159113324811538...  0.362869  0.0  0.864407   
1  1.2.826.0.1.3680043.8.498.11771393824519892797...  0.000000  0.0  0.177998   
2  1.2.826.0.1.3680043.8.498.13267780356245120052...  0.141361  0.0  0.000000   
3  1.2.826.0.1.3680043.8.498.16060119389060497136...  0.220408  0.0  0.000000   
4  1.2.826.0.1.3680043.8.498.17844546765907321649...  0.471299  0.0  0.000000   
5  1.2.826.0.1.3680043.8.498.30246079718471552972...  0.000000  0.0  0.820051   

          3         4         5         6         7         8         9  \
0  1.000000  0.885304  0.859551  1.000000  0.397069  0.790240  0.666667   
1  0.073171  0.238212  0.000000  0.000000  1.000000  0.000000  0.000000   
2  0.259146  0.000000  0.000000  0.000000  0.317355  0.000000  0.448980   
3  0.465753  0.000000  0.200000  0.605699  0.589144  0.361299  0.171875   
4  0.554945  0.288744  0.000000  0.456894  0.907891  0.52

In [17]:
# Depth 2 ensemble
# fluid sensitive and fat suppression can either be 0 or 1
ensemble = []

for p in planes:
    for i in range(2):
        ensemble.append(LogRegModel(p, i, i, n_comp=7))

scores = Model.get_ensemble_auc_score(ensemble, 2)

print(f"Model Score: {np.mean(scores)}")
for i in range(len(target_columns)):
    print(f"\t{target_columns[i]}: {scores[i]}")

(26, 18, 512, 512)
(23, 18, 512, 512)
(7, 18, 512, 512)


/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 5 is present in all training examples.
  warnings.warn(
/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 9 is present in all training examples.
  warnings.warn(
/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 11 is present in all training examples.
  warnings.warn(
/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


(22, 18, 512, 512)
(19, 18, 512, 512)


/home/zero/venvs/ds-venv/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


(23, 18, 512, 512)
                                               index        0    1         2  \
0  1.2.826.0.1.3680043.8.498.10306159113324811538...  0.12938  NaN  0.641791   
1  1.2.826.0.1.3680043.8.498.11771393824519892797...  0.00000  0.0  0.000000   
2  1.2.826.0.1.3680043.8.498.13267780356245120052...  0.00000  0.0  0.000000   
3  1.2.826.0.1.3680043.8.498.16060119389060497136...      NaN  NaN       NaN   
4  1.2.826.0.1.3680043.8.498.17844546765907321649...  0.16632  NaN  0.000000   
5  1.2.826.0.1.3680043.8.498.30246079718471552972...  0.00000  NaN  1.000000   

          3         4         5         6         7         8         9   10  \
0  0.857605  1.000000  0.477523  0.865088  0.596639  0.687500  0.583916  0.0   
1  0.481481  0.163325  0.000000  0.099080  0.352078  0.646341  0.000000  0.0   
2  0.522222  0.000000  0.000000  0.272470  0.168704  0.000000  0.000000  0.0   
3       NaN       NaN       NaN       NaN       NaN       NaN       NaN  NaN   
4  0.620553  0.09469

ValueError: Input contains NaN.

In [10]:
full_df

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10095687747295410396...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.2.826.0.1.3680043.8.498.10791858932231443712...,0,0,Sagittal
1,1.2.826.0.1.3680043.8.498.10095687747295410396...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.2.826.0.1.3680043.8.498.12451507871565834868...,0,0,Coronal
2,1.2.826.0.1.3680043.8.498.10095687747295410396...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.2.826.0.1.3680043.8.498.21497837050858589852...,1,1,Coronal
3,1.2.826.0.1.3680043.8.498.10095687747295410396...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.2.826.0.1.3680043.8.498.37372672584387115708...,1,1,Sagittal
4,1.2.826.0.1.3680043.8.498.10095687747295410396...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.2.826.0.1.3680043.8.498.67495435033389756963...,1,1,Axial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,1.2.826.0.1.3680043.8.498.30246079718471552972...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.2.826.0.1.3680043.8.498.10125030789765676430...,1,1,Coronal
162,1.2.826.0.1.3680043.8.498.30246079718471552972...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.2.826.0.1.3680043.8.498.12908352013592575797...,1,1,Axial
163,1.2.826.0.1.3680043.8.498.30246079718471552972...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.2.826.0.1.3680043.8.498.13364655615747190896...,0,0,Sagittal
164,1.2.826.0.1.3680043.8.498.30246079718471552972...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.2.826.0.1.3680043.8.498.73098240504448530683...,1,1,Sagittal


In [11]:
test_series_df

,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.11580656442259111255...,0,0,Axial
1,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.17811502614030631664...,0,0,Sagittal
2,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.30565395595045942404...,0,0,Sagittal
3,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.32856494541816845805...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10047035057544427318...,1.2.826.0.1.3680043.8.498.44334485654554877495...,1,1,Axial
5,1.2.826.0.1.3680043.8.498.10062861783145312629...,1.2.826.0.1.3680043.8.498.10543708582570884867...,1,1,Sagittal
6,1.2.826.0.1.3680043.8.498.10062861783145312629...,1.2.826.0.1.3680043.8.498.10818107180155454609...,1,1,Axial
7,1.2.826.0.1.3680043.8.498.10062861783145312629...,1.2.826.0.1.3680043.8.498.34851887739080902195...,1,1,Sagittal
8,1.2.826.0.1.3680043.8.498.10062861783145312629...,1.2.826.0.1.3680043.8.498.49057071886338907232...,0,0,Sagittal
9,1.2.826.0.1.3680043.8.498.10062861783145312629...,1.2.826.0.1.3680043.8.498.49397095733611495516...,1,1,Coronal
